In [ ]:
import torch
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, DatasetDict, Audio
from transformers import WhisperFeatureExtractor, Seq2SeqTrainer, WhisperTokenizer, WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments

## Load Data

In [ ]:
common_voice = DatasetDict()
common_voice["train"] = load_dataset("mozilla-foundation/common_voice_17_0", "am", split="train+validation", trust_remote_code=True)
common_voice["test"] = load_dataset("mozilla-foundation/common_voice_17_0", "am", split="test", trust_remote_code=True)

In [ ]:
# Remove unusde columns
common_voice = common_voice.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"])

print(common_voice)

In [ ]:
# Resample Audio 48 - 16kHz
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
#  Extracts log-Mel spectrogram features 
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    batch["lables"] = tokenizer(batch["sentence"]).input_ids

    return batch

## Prepare Feature Extractor, Tokenizer and Input Data

In [ ]:
# Feature Extractor
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

# Tokenizer
tokenizer = WhisperTokenizer.from_pretrained(
    "openia/whisper-small",
    language="Amharic",
    task="transcribe"
)

# Input Data
common_voice = common_voice.map(prepare_dataset, remove_columns=common_voice.column_names["train"], num_proc=2)

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Amharic", task="transcribe")

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.generation_config.language = "amharic"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
# Evaluation
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

### Training Argument 

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./dir",
    report_to=[],
    per_device_train_batch_size=8,
    learning_rate=1e-5,
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="no",
    save_steps=500,
    logging_steps=25,
)

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # tokenizer=processor.feature_extractor,
    processing_class=processor
)

In [ ]:
# Train
trainer.train()